In [19]:
import pandas as pd
import numpy as np
import pymysql
from sqlalchemy import create_engine,text


In [20]:

df = pd.read_csv("HR_Employee_Attrition.csv")

In [21]:


# 3. Initial inspection (rows, columns, info)
print(df.head())
print(df.info())
print(df.shape)

# 4. Standardize/clean column names (strip spaces, lower case)
df.columns = [col.strip().replace(" ", "_").lower() for col in df.columns]

# 5. Remove duplicate rows
print("Duplicate rows:", df.duplicated().sum())
df = df.drop_duplicates()

# 6. Identify missing values & weird NAs
print(df.isnull().sum())
print(df.isin(['NA', 'N/A', ' ', '']).sum())

# 7. Replace missing/blank/NA values with np.nan
df.replace(['NA', 'N/A', ' ', ''], np.nan, inplace=True)

# 8. Impute/fill missing values for each column (numeric: median, categorical: mode)
for col in df.columns:
    if df[col].isnull().sum() > 0:
        if df[col].dtype == "O":
            df[col].fillna(df[col].mode()[0], inplace=True)
        else:
            df[col].fillna(df[col].median(), inplace=True)

# 9. Convert object columns to numeric IF possible (auto-detect)
for col in df.columns:
    if df[col].dtype == "object":
        try:
            df[col] = pd.to_numeric(df[col], errors="raise")
            print(f"{col} converted to numeric.")
        except:
            pass  # Not numeric, leave as object

# 10. Remove irrelevant columns with single unique value (waste for analysis)
for col in ['employeecount', 'standardhours', 'over18']:
    if col in df.columns and df[col].nunique() == 1:
        df.drop(col, axis=1, inplace=True)
        print(f"{col} dropped.")

# 11. Strip trailing spaces from strings (object columns)
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].str.strip()

# 12. Double check types and uniques (for plotting/groupby)
print(df.info())
print("Unique value count per column:\n", df.nunique())

# 13. Categorical value_counts for EDA
for col in df.select_dtypes(include="object").columns:
    print(f"\nValue counts for {col}:\n", df[col].value_counts())

# 14. Export cleaned data for EDA/analysis
df.to_csv("hr_attrition_cleaned_data.csv", index=False)
print("Cleaned file saved as 'hr_attrition_cleaned_data.csv'.")


   Age Attrition     BusinessTravel  DailyRate              Department  \
0   41       Yes      Travel_Rarely       1102                   Sales   
1   49        No  Travel_Frequently        279  Research & Development   
2   37       Yes      Travel_Rarely       1373  Research & Development   
3   33        No  Travel_Frequently       1392  Research & Development   
4   27        No      Travel_Rarely        591  Research & Development   

   DistanceFromHome  Education EducationField  EmployeeCount  EmployeeNumber  \
0                 1          2  Life Sciences              1               1   
1                 8          1  Life Sciences              1               2   
2                 2          2          Other              1               4   
3                 3          4  Life Sciences              1               5   
4                 2          1        Medical              1               7   

   ...  RelationshipSatisfaction StandardHours  StockOptionLevel  \
0  ...

In [22]:
df.shape

(1470, 32)

In [23]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1470 entries, 0 to 1469
Data columns (total 32 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   age                       1470 non-null   int64 
 1   attrition                 1470 non-null   object
 2   businesstravel            1470 non-null   object
 3   dailyrate                 1470 non-null   int64 
 4   department                1470 non-null   object
 5   distancefromhome          1470 non-null   int64 
 6   education                 1470 non-null   int64 
 7   educationfield            1470 non-null   object
 8   employeenumber            1470 non-null   int64 
 9   environmentsatisfaction   1470 non-null   int64 
 10  gender                    1470 non-null   object
 11  hourlyrate                1470 non-null   int64 
 12  jobinvolvement            1470 non-null   int64 
 13  joblevel                  1470 non-null   int64 
 14  jobrole                 

In [24]:
import pandas as pd
from sqlalchemy import create_engine

# 1. Load your cleaned dataset
df_clean= pd.read_csv("hr_attrition_cleaned_data.csv")

# 2. Create connection engine to MySQL
username = "root"    
password = "pricass00"    
host = "localhost"
database = "hr_analytics"           

connection_string = f"mysql+pymysql://{username}:{password}@{host}/{database}"
engine = create_engine(connection_string)

# 3. Upload DataFrame to a MySQL table called 'attrition'; overwrite if exists
df_clean.to_sql("employee_attrition", engine, if_exists="replace", index=False)
print("Cleaned dataset loaded to MySQL table 'attrition' successfully!")


Cleaned dataset loaded to MySQL table 'attrition' successfully!


In [25]:
df_clean.shape

(1470, 32)

In [26]:
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1470 entries, 0 to 1469
Data columns (total 32 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   age                       1470 non-null   int64 
 1   attrition                 1470 non-null   object
 2   businesstravel            1470 non-null   object
 3   dailyrate                 1470 non-null   int64 
 4   department                1470 non-null   object
 5   distancefromhome          1470 non-null   int64 
 6   education                 1470 non-null   int64 
 7   educationfield            1470 non-null   object
 8   employeenumber            1470 non-null   int64 
 9   environmentsatisfaction   1470 non-null   int64 
 10  gender                    1470 non-null   object
 11  hourlyrate                1470 non-null   int64 
 12  jobinvolvement            1470 non-null   int64 
 13  joblevel                  1470 non-null   int64 
 14  jobrole                 

In [27]:
df_sql = pd.read_sql("SELECT * FROM employee_attrition", con=engine)


In [28]:
df_sql.shape

(1470, 32)

In [29]:

df_sql.head()


,age,attrition,businesstravel,dailyrate,department,distancefromhome,education,educationfield,employeenumber,environmentsatisfaction,...,performancerating,relationshipsatisfaction,stockoptionlevel,totalworkingyears,trainingtimeslastyear,worklifebalance,yearsatcompany,yearsincurrentrole,yearssincelastpromotion,yearswithcurrmanager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,2,...,3,1,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,2,3,...,4,4,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,4,4,...,3,2,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,5,4,...,3,3,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,7,1,...,3,4,1,6,3,3,2,2,2,2


In [30]:
 
print(df_sql.info())     

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1470 entries, 0 to 1469
Data columns (total 32 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   age                       1470 non-null   int64 
 1   attrition                 1470 non-null   object
 2   businesstravel            1470 non-null   object
 3   dailyrate                 1470 non-null   int64 
 4   department                1470 non-null   object
 5   distancefromhome          1470 non-null   int64 
 6   education                 1470 non-null   int64 
 7   educationfield            1470 non-null   object
 8   employeenumber            1470 non-null   int64 
 9   environmentsatisfaction   1470 non-null   int64 
 10  gender                    1470 non-null   object
 11  hourlyrate                1470 non-null   int64 
 12  jobinvolvement            1470 non-null   int64 
 13  joblevel                  1470 non-null   int64 
 14  jobrole                 